# TFDV Lab 6: Telco Churn Dataset

This version of the lab uses a Telco Churn dataset instead of the Adult/Census dataset. The goal is the same: generate statistics, infer a schema, detect anomalies, revise the schema, and inspect slices.

## Imports

In [ ]:
import tensorflow as tf
import tensorflow_data_validation as tfdv
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow_metadata.proto.v0 import schema_pb2
from tensorflow_data_validation.utils import slicing_util
from tensorflow_metadata.proto.v0.statistics_pb2 import DatasetFeatureStatisticsList

from util import add_telco_extra_rows

print('TFDV Version:', tfdv.__version__)
print('TensorFlow Version:', tf.__version__)

## Load the Telco Churn dataset

In [ ]:
df = pd.read_csv('data/telco_churn.csv')
train_df, eval_df = train_test_split(df, test_size=0.2, shuffle=False)

print('Train shape:', train_df.shape)
print('Eval shape:', eval_df.shape)
train_df.head()

## Add malformed rows to the evaluation split

These extra rows simulate drift and data quality issues such as new categories, invalid numeric ranges, and missing labels.

In [ ]:
eval_df = add_telco_extra_rows(eval_df)
eval_df.tail(4)

## Generate training statistics and infer schema

In [ ]:
train_stats = tfdv.generate_statistics_from_dataframe(train_df)
tfdv.visualize_statistics(train_stats)

In [ ]:
schema = tfdv.infer_schema(statistics=train_stats)
tfdv.display_schema(schema)

## Compare evaluation statistics with training statistics

In [ ]:
eval_stats = tfdv.generate_statistics_from_dataframe(eval_df)
tfdv.visualize_statistics(
    lhs_statistics=eval_stats,
    rhs_statistics=train_stats,
    lhs_name='EVAL_DATASET',
    rhs_name='TRAIN_DATASET'
)

## Clean obvious numeric errors

In [ ]:
eval_df = eval_df[eval_df['tenure'] >= 0]
eval_df = eval_df[eval_df['tenure'] <= 72]
eval_df = eval_df[eval_df['MonthlyCharges'] <= 150]
eval_df = eval_df[eval_df['TotalCharges'] >= 0]
eval_df = eval_df.dropna(subset=['Churn'])

eval_stats = tfdv.generate_statistics_from_dataframe(eval_df)
tfdv.visualize_statistics(
    lhs_statistics=eval_stats,
    rhs_statistics=train_stats,
    lhs_name='CLEAN_EVAL_DATASET',
    rhs_name='TRAIN_DATASET'
)

## Validate anomalies

In [ ]:
anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)
tfdv.display_anomalies(anomalies)

## Revise the schema

We will allow a new gender value and relax domain mass for fields where new-but-plausible payment and internet-service categories may appear.

In [ ]:
gender_domain = tfdv.get_domain(schema, 'gender')
gender_domain.value.append('NonBinary')

payment_feature = tfdv.get_feature(schema, 'PaymentMethod')
payment_feature.distribution_constraints.min_domain_mass = 0.9

internet_feature = tfdv.get_feature(schema, 'InternetService')
internet_feature.distribution_constraints.min_domain_mass = 0.9

tfdv.set_domain(schema, 'tenure', schema_pb2.IntDomain(name='tenure', min=0, max=72))
tfdv.set_domain(schema, 'MonthlyCharges', schema_pb2.FloatDomain(name='MonthlyCharges', min=15.0, max=130.0))
tfdv.set_domain(schema, 'TotalCharges', schema_pb2.FloatDomain(name='TotalCharges', min=0.0, max=9000.0))

tfdv.display_schema(schema)

In [ ]:
updated_anomalies = tfdv.validate_statistics(eval_stats, schema)
tfdv.display_anomalies(updated_anomalies)

## Examine slices

We will compare `gender` slices first, then compare a more specific `gender + Contract` slice.

In [ ]:
CSV_PATH = 'telco_slice_sample.csv'
train_df.to_csv(CSV_PATH, index=False)

slice_fn = slicing_util.get_feature_value_slicer(features={'gender': None})
slice_stats_options = tfdv.StatsOptions(
    schema=schema,
    experimental_slice_functions=[slice_fn],
    infer_type_from_schema=True
)
sliced_stats = tfdv.generate_statistics_from_csv(CSV_PATH, stats_options=slice_stats_options)
print([dataset.name for dataset in sliced_stats.datasets])

In [ ]:
female_stats_list = DatasetFeatureStatisticsList()
female_stats_list.datasets.extend([sliced_stats.datasets[1]])

male_stats_list = DatasetFeatureStatisticsList()
male_stats_list.datasets.extend([sliced_stats.datasets[2]])

tfdv.visualize_statistics(
    lhs_statistics=female_stats_list,
    rhs_statistics=male_stats_list,
    lhs_name=sliced_stats.datasets[1].name,
    rhs_name=sliced_stats.datasets[2].name
)

In [ ]:
slice_fn = slicing_util.get_feature_value_slicer(
    features={
        'gender': [b'Female', b'Male'],
        'Contract': [b'Month-to-month', b'Two year']
    }
)

slice_stats_options = tfdv.StatsOptions(
    schema=schema,
    experimental_slice_functions=[slice_fn],
    infer_type_from_schema=True
)
sliced_stats = tfdv.generate_statistics_from_csv(
    data_location=CSV_PATH,
    stats_options=slice_stats_options
)
print([dataset.name for dataset in sliced_stats.datasets])

In [ ]:
female_month_stats_list = DatasetFeatureStatisticsList()
female_month_stats_list.datasets.extend([sliced_stats.datasets[1]])

male_two_year_stats_list = DatasetFeatureStatisticsList()
male_two_year_stats_list.datasets.extend([sliced_stats.datasets[-1]])

tfdv.visualize_statistics(
    lhs_statistics=female_month_stats_list,
    rhs_statistics=male_two_year_stats_list,
    lhs_name=sliced_stats.datasets[1].name,
    rhs_name=sliced_stats.datasets[-1].name
)

## Wrap up

This custom version keeps the original TFDV workflow but applies it to customer churn data. The main changes are the dataset, the anomaly rules, and the slice definitions.